# Module 4 — Information Extraction

**What you are doing today:** your bot already works out *what* the customer
wants. Today it pulls out *which thing* — the order number, the product.

`ORD-48210 where??` is classified perfectly by your Module 3 bot, and the reply
it can write is still *"let me check your order"* — which order? Today that
stops.

| Part | What you do |
| --- | --- |
| **A — Follow along** | Run each cell, read the *Notice* line. Nothing to write. |
| **B — Your turn** | Six tasks. Now you write the code. |

Two tools, and they are deliberately different:

- **A pattern**, for anything with a fixed **shape** — `ORD-48210`, a phone
  number, a postcode.
- **A catalogue lookup**, for anything with **no shape** — product names. The
  shop already knows every product it sells, so look them up.

**Have your 10 collected messages open.** You need them in task B5.

Work in pairs. Swap who types between the two parts.


---
# Part A — Follow along

Seven short steps, A1 to A7, after one setup cell. **Run each cell, then read
the *Notice* line under it.** You do not write anything in this part.

By A5 you will find a bug your own code has had since Module 2. By A7 the bot
reports, for every message, exactly which details it has and which it is
missing.


### Setup — run this first

This rebuilds everything from Modules 1, 2 and 3 in one cell — `clean_text`,
`fix_typos` and `classify` — so nobody is stuck because of missing homework.
Nothing here is new. Run it and move on.

The first run takes 15–20 seconds while Colab connects and downloads the
stopword list.


In [ ]:
import string, re
from difflib import get_close_matches

# --- Module 2: the stopword list, minus the nine question words we kept ---
try:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    NLTK_STOP = set(stopwords.words("english"))
except Exception:
    NLTK_STOP = set("""i me my we our you your he she it its they them this that
    is are was were be been being have has had do does did a an the and but if or
    because as of at by for with about into through during to from in out on off
    then once here there all any both each few more most other some such no nor
    not only own same so than too very s t can will just don should now""".split())

KEEP = {"where", "when", "how", "why", "what", "not", "no", "before", "after"}
STOP = NLTK_STOP - KEEP

# --- Module 3: the intent word lists ---
INTENTS = {
    "check_order_status": {"where", "order", "parcel", "package", "delivery",
                           "tracking", "status", "receive", "arrived"},
    "return_item":        {"return", "refund", "exchange", "back", "broken",
                           "wrong", "size", "shoes", "bag"},
    "delivery_time":      {"when", "how", "long", "time", "days", "arrive",
                           "deliver", "shipping", "fast"},
    "greeting":           {"hello", "hi", "hey", "morning", "afternoon",
                           "evening", "greetings"},
    "goodbye":            {"bye", "goodbye", "thanks", "thank", "tq", "ok",
                           "cheers"},
}
KEYWORDS = sorted({w for wl in INTENTS.values() for w in wl})

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return [w for w in text.split() if w not in STOP]

def fix_typos(words, cutoff=0.7):
    out = []
    for w in words:
        m = get_close_matches(w, KEYWORDS, n=1, cutoff=cutoff)
        out.append(m[0] if m else w)
    return out

def classify(text, cutoff=1):          # cut-off 1: the dial Module 3 ended on
    words = fix_typos(clean_text(text))
    scores = {i: sum(1 for w in words if w in wl) for i, wl in INTENTS.items()}
    best = max(scores, key=scores.get)
    if scores[best] < cutoff:
        return "fallback", scores
    return best, scores

print("Modules 1-3 rebuilt.")
print("classify('where is my order') ->", classify("where is my order")[0])

**Notice:** `classify('where is my order') -> check_order_status`. If that
printed, your Module 3 bot is alive inside this notebook and you can start.


### A1 — The intent is only half the job

Two messages. One of them hands the bot the order number. Watch what the
classifier does with that.


In [ ]:
for m in ["where is my order", "ORD-48210 where??"]:
    intent, scores = classify(m)
    print(f"{m:26} -> {intent:20} score {max(scores.values())}")

**Notice:** both come back `check_order_status`. The second customer typed the
one piece of information that would let the bot actually look something up, and
`classify` threw it away — `ord48210` is not in any word list, so it scored
nothing.

The intent tells the bot **what to do**. It does not tell it **what to do it
to**. That second half is today.


### A2 — The simplest method that works: a list

A shop knows its own products. There are only so many. So to find a product name
in a message, do not be clever — check the catalogue.

Fifteen products, and `if item in text` for each one.


In [ ]:
CATALOGUE = ["blue shoes", "black shoes", "running shoes", "shoes", "shoelaces",
             "red bag", "school bag", "laptop bag", "water bottle",
             "pencil case", "blue shirt", "white shirt", "hoodie", "cap", "socks"]

def find_product(text):
    text = text.lower()
    for item in CATALOGUE:
        if item in text:
            return item
    return None

for m in ["i want to return the blue shoes",
          "can i return the shoes i bought last week ah",
          "need to change my water bottle ORD-77031",
          "my shoelaces broke",
          "the bag is broken, i want exchange"]:
    print(f"{m:46} -> {find_product(m)}")

**Notice:** four out of five. Nine lines of code, no library, and it can never
invent a product the shop does not sell — that last part is the real reason to
start here.

The fifth one is the interesting row. `the bag is broken` returns `None`,
because the catalogue sells a **red bag**, a **school bag** and a **laptop bag**,
and the customer just said *bag*. The shop's own words and the customer's words
are not the same words.


### A3 — Customers cannot spell, so reuse Module 2

`blu shirt` is not in the catalogue. `get_close_matches` — the same tool that
fixed typos in Module 2 — will find it anyway.

This version takes the **longest** matching catalogue entry rather than the
first, and only close-matches when a plain lookup finds nothing.


In [ ]:
def find_product_longest(text):
    text = text.lower()
    hits = [item for item in CATALOGUE if item in text]
    return max(hits, key=len) if hits else None

def find_product_close(text, cutoff=0.7):
    hit = find_product_longest(text)
    if hit:
        return hit
    words = text.lower().split()
    chunks = [" ".join(words[i:i+2]) for i in range(len(words)-1)] + words
    for c in chunks:
        m = get_close_matches(c, CATALOGUE, n=1, cutoff=cutoff)
        if m:
            return m[0]
    return None

for m in ["i want to return the blu shirt", "my blak shoes are wrong size",
          "wheres my wter bottle", "the bag is broken, i want exchange",
          "i want to return the blue shoes"]:
    print(f"{m:36} plain={str(find_product_longest(m)):14} close={find_product_close(m)}")

**Notice:** `blu shirt` becomes `blue shirt` and `wter bottle` becomes
`water bottle`. Two spelling mistakes rescued for free, because Module 2 already
built the tool.

Now look at row four again. `the bag is broken` now returns **`red bag`** — and
the customer never said red. The close matcher will always return *something* if
you let it. Hold onto that; it is task B3.


### A4 — Things with a shape: a pattern

A product name has no shape. An order number does: three capital letters, a
hyphen, five digits. Every single one.

Four pieces of notation is all you need.

| Piece | Means | Example |
| --- | --- | --- |
| `\d` | any digit | `\d` matches `7` |
| `{5}` | exactly five of them | `\d{5}` matches `48210` |
| `+` | one or more | `\d+` matches `4` **or** `48210` |
| `[A-Z]` | any capital letter | `[A-Z]{3}` matches `ORD` |

So the order number is `ORD-\d{5}`. `re.search` looks for it anywhere in the
text and `.group()` hands back what it found.


In [ ]:
def find_order_id(text):
    match = re.search(r"ORD-\d{5}", text.upper())
    return match.group() if match else None

tests = ["ORD-48210 where??", "my order ORD-48210", "ord-48210",
         "ORD-4821", "ORDER 48210", "ORD-482105", "no id here"]

for t in tests:
    print(f"{t:22} -> {find_order_id(t)}")

**Notice:** six of these behave exactly as you would expect. `ord-48210` matches
because of `text.upper()`; `ORD-4821` and `ORDER 48210` correctly find nothing.

**`ORD-482105` does not behave.** It returns `ORD-48210` — it chopped the last
digit off and reported a different order. Do not fix it yet. That is task B1.


### A5 — Where do you run this? (the Module 2 bill arrives)

Module 2's `clean_text` strips punctuation. Watch what that does to an order
number.


In [ ]:
msg = "ORD-48210 where??"

print("raw message   :", repr(msg))
print("after cleaning:", clean_text(msg))
print()
print("find_order_id(raw)     ->", find_order_id(msg))
print("find_order_id(cleaned) ->", find_order_id(" ".join(clean_text(msg))))
print()
print("and the same thing to a hash:", clean_text("order#4521 still not here"))

**Notice:** `ORD-48210` became `ord48210`. The hyphen is punctuation, so
Module 2 deleted it — and with it, the shape the pattern was looking for.
`order#4521` became `order4521` the same way.

Nothing errored. The bot just quietly stops finding order numbers.

**So extraction runs on the raw message, before cleaning.** Classification runs
on the cleaned one. Two stages, two different inputs, and that is not a bodge —
it is what every real pipeline does.


### A6 — The other option: spare those characters

You could instead teach the cleaner to keep `-` and `#`. Both fixes are legitimate.


In [ ]:
KEEP_CHARS = "-#"
DROP = "".join(c for c in string.punctuation if c not in KEEP_CHARS)

def clean_keep_ids(text):
    text = text.lower().translate(str.maketrans("", "", DROP))
    return [w for w in text.split() if w not in STOP]

print("old cleaner :", clean_text("ORD-48210 where??"))
print("new cleaner :", clean_keep_ids("ORD-48210 where??"))
print()
print("old cleaner :", clean_text("order#4521 still not here"))
print("new cleaner :", clean_keep_ids("order#4521 still not here"))

**Notice:** `['ord-48210', 'where']` — the ID survives. But now `fix_typos` and
the intent scorer have to cope with a word containing a hyphen, and every other
message in the course keeps punctuation it did not ask for.

For the rest of today we use the first fix: **extract from raw, classify from
cleaned.** It changes nothing downstream.


### A7 — Wire the details into the bot

Each intent needs different details. `return_item` needs to know which item
**and** which order. `greeting` needs nothing.

That list is the whole idea, and it is a dictionary.


In [ ]:
def find_order_id(text):                      # A4, now with \b
    match = re.search(r"ORD-\d{5}\b", text.upper())
    return match.group() if match else None

REQUIRED = {
    "check_order_status": ["order_id"],
    "return_item":        ["product", "order_id"],
    "delivery_time":      [],
    "greeting":           [],
    "goodbye":            [],
    "fallback":           [],
}

def extract(text):
    return {"order_id": find_order_id(text),
            "product":  find_product_close(text)}

def handle(text):
    intent, _ = classify(text)                # cleaned text
    slots = extract(text)                     # RAW text
    missing = [s for s in REQUIRED.get(intent, []) if not slots.get(s)]
    print("Customer:", text)
    print("  Intent: ", intent)
    print("  Found:  ", {k: v for k, v in slots.items() if v})
    print("  Missing:", missing)
    print()

for t in ["ORD-48210 where??", "i want to return the blue shoes",
          "where is my order", "can i return the shoes i bought last week ah"]:
    handle(t)

**Notice:** `ORD-48210 where??` now reports `Missing: []` — the bot has
everything it needs to look that order up. Compare it with `where is my order`,
which is the same intent and is `Missing: ['order_id']`.

And look at the return request: it found `blue shoes` and it is missing
`order_id`. It says so, and then it **does nothing about it**. That is
deliberate, it is not your code being broken, and it is where Module 5 starts.


---
# Part B — Your turn

Six tasks. The `___` blanks are the idea; everything around them is given.

Most pairs will not finish all six. **B1, B2 and B3 are the ones that matter** —
do those properly rather than rushing to B6.


### B1 — Predict, then run

Before you run anything, fill in the middle column. Write what you think
`find_order_id_v1` (the A4 version, `ORD-\d{5}`, no `\b`) returns.

| Input | Your prediction | What it actually did |
| --- | --- | --- |
| `ORD-48210` | | |
| `ord-48210` | | |
| `ORD-4821` | | |
| `ORDER 48210` | | |
| `ORD-482105` | | |
| `ORD-48210 and ORD-51774` | | |

Fill in the predictions. Then run.


In [ ]:
def find_order_id_v1(text):
    match = re.search(r"ORD-\d{5}", text.upper())
    return match.group() if match else None

for t in ["ORD-48210", "ord-48210", "ORD-4821", "ORDER 48210", "ORD-482105",
          "ORD-48210 and ORD-51774"]:
    print(f"{t:26} -> {find_order_id_v1(t)}")

Two rows should have surprised you.

**`ORD-482105` returned `ORD-48210`.** `\d{5}` takes five digits and stops. It
does not care that a sixth digit was sitting right there. The bot did not fail —
it confidently reported *a different order that probably exists*.

**`ORD-48210 and ORD-51774` returned only the first one.** `re.search` finds
**one** match. Two order numbers in a message and the second is invisible.

Now fix the first one. `\b` means *and the thing must end here*:

```python
re.search(r"ORD-\d{5}\b", text.upper())
```

Run the same six through the fixed version and check that `ORD-482105` now
returns `None`.


In [ ]:
def find_order_id_v2(text):
    match = re.search(r"ORD-\d{5}___", text.upper())     # <-- add the \b
    return match.group() if match else None

for t in ["ORD-48210", "ord-48210", "ORD-4821", "ORDER 48210", "ORD-482105",
          "ORD-48210 and ORD-51774"]:
    print(f"{t:26} -> {find_order_id_v2(t)}")

### B2 — Find the bug

A classmate reordered the catalogue so the short names come first. They say it
is tidier. Run their version.

Nothing crashes. The answers are wrong anyway.


In [ ]:
CAT_TIDY = ["shoes", "blue shoes", "black shoes", "running shoes", "shoelaces",
            "red bag", "school bag", "laptop bag", "water bottle",
            "pencil case", "blue shirt", "white shirt", "hoodie", "cap", "socks"]

def find_product_tidy(text):
    text = text.lower()
    for item in CAT_TIDY:
        if item in text:
            return item
    return None

for m in ["i want to return the blue shoes", "my black shoes are the wrong size"]:
    print(f"{m:40} tidy={str(find_product_tidy(m)):14} ours={find_product_close(m)}")

**Write down, in one sentence, what went wrong.** Not *"the code is wrong"* —
say which line caused it.

Then answer this: the A3 version is not clever, it just takes the **longest**
match instead of the first. Why does longest work here, and what would break it?


### B3 — Turn the dial: how greedy should a pattern be?

`ORD-\d{5}\b` is strict. You could loosen it. Below are three patterns, from
strict to greedy, run against fifteen messages.

**Before you run it:** which one do you expect to find the most order IDs? Which
one do you expect to be *right* most often?

Fill in the third pattern — make it as greedy as possible: one or more digits,
nothing else.


In [ ]:
MESSAGES = [
    "ORD-48210 where??", "hi, my order still havent arrive",
    "i want to return the blue shoes",
    "can i return the shoes i bought last week ah", "my parcel not here yet",
    "order ORD-51774 not received", "tracking pls", "ord-48210 status?",
    "how long to deliver to penang", "the bag is broken, i want exchange",
    "ORDER 48210 where", "wrong size how", "i wan 2 chk my ordr",
    "ORD-482105 is my order number", "need to change my water bottle ORD-77031",
]

PATTERNS = [r"ORD-\d{5}\b", r"[A-Z]{3}-\d+", r"___"]     # <-- the greedy one

for p in PATTERNS:
    hits = [(m, re.search(p, m.upper()).group()) for m in MESSAGES
            if re.search(p, m.upper())]
    print(f"\n{p:16}  {len(hits)} / 15 messages matched")
    for m, g in hits:
        print(f"      {m:46} -> {g}")

Now fill this in from what printed:

| Pattern | Matches | How many are actually an order ID? |
| --- | --- | --- |
| `ORD-\d{5}\b` | | |
| `[A-Z]{3}-\d+` | | |
| `\d+` | | |

**The question that matters:** the greedy pattern finds the most. Is it the best
one? Write one sentence saying what a customer would experience if the bot used
it.


### B4 — Write your own pattern

A Malaysian mobile number looks like `012-3456789`: `01`, one more digit, a
hyphen, then seven or eight digits.

You have `\d`, `{n}`, `{n,m}` and `[A-Z]`. Write the pattern.

Half the test strings below **should not match**. A pattern that matches
everything is a broken pattern.


In [ ]:
def find_phone(text):
    m = re.search(r"___", text)               # <-- your pattern
    return m.group() if m else None

ptests = ["call me at 012-3456789", "my number is 011-12345678", "016-2233445",
          "whatsapp +6012-3456789", "0123456789", "phone 03-79551234",
          "ORD-48210 where??", "i need it by 03/05", "no number here",
          "01-2345678"]

for t in ptests:
    print(f"{t:28} -> {find_phone(t)}")

Check your six rejections. If your pattern matched `i need it by 03/05` or
`ORD-48210 where??`, it is too greedy — go back and tighten it.


### B5 — Your own ten messages

Paste your ten collected messages into the list. Run `extract` over them and
count three things: how many gave up an order ID, how many gave up a product,
and how many gave up **nothing at all**.

That last number is the one to write down.


In [ ]:
my_messages = [
    "ORD-48210 where??",
    "i want to return the blue shoes",
    "___",
    "___",
    "___",
    # ... all 10
]

my_messages = [m for m in my_messages if m != "___"]

n_id = n_prod = n_none = 0
for i, m in enumerate(my_messages, 1):
    s = extract(m)
    n_id += bool(s["order_id"])
    n_prod += bool(s["product"])
    n_none += not (s["order_id"] or s["product"])
    print(f"{i:2}. {m[:44]:46} {str(s['order_id']):12} {s['product']}")

print()
print(f"order id found : {n_id} / {len(my_messages)}")
print(f"product found  : {n_prod} / {len(my_messages)}")
print(f"nothing at all : {n_none} / {len(my_messages)}")

**Then answer:** pick one message that gave up nothing. Is that a bug in your
extractor, or did the customer genuinely not say anything to extract?

They are different problems and only one of them is yours to fix.


### B6 — The one you are not fixing today

Run this. Read the two exchanges as a conversation.


In [ ]:
handle("i want to return the blue shoes")

print("Bot: Which item would you like to return?")
print()

handle("the blue one")

**Write down what the bot should have done between those two turns, and why it
cannot.**

Do not try to fix it. The fix is the whole of Module 5.


---
## Stretch — what professionals actually use

Optional. Nothing in Module 5 assumes you did this.

Patterns work for fixed shapes. They are useless for *"Ahmad"*, *"Penang"* or
*"last Tuesday"* — none of those has a shape. Real systems use a trained model
for that, and the job has a name: **named entity recognition**, or NER.

The cell below installs **spaCy** and runs it. You do not need to understand the
model. Just look at what it pulls out.


In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import spacy
nlp = spacy.load("en_core_web_sm")

samples = [
    "I want to return the blue shoes I ordered on 3 May",
    "how long to deliver to penang",
    "ORD-48210 where??",
    "Ahmad here, my parcel to Penang is late",
]

for s in samples:
    doc = nlp(s)
    print(f"{s}")
    print(f"   -> {[(e.text, e.label_) for e in doc.ents]}")
    print()

**Then answer both:**

1. Write down **two** things it got right and **one** it got wrong.
2. The shop has a product list with fifteen items on it. Why would it still use
   that list for finding product names, instead of this?


---
## Homework

Add **two new entity types** your store bot would need. Pick from: **postcode**,
**quantity**, **colour**, **size** — or invent your own.

For each one:

1. Write the extractor. A pattern if it has a shape, a list if it does not.
2. Test it on **five messages you invent**.
3. **One of those five must be a near-miss that your extractor should reject** —
   something that looks almost right and is not.

Bring the near-miss that fooled you. Those are worth more than the four that
worked.

**A worked start, so you know the shape of the answer:**

```python
def find_postcode(text):                 # Malaysian postcodes are five digits
    m = re.search(r"\d{5}", text)
    return m.group() if m else None

print(find_postcode("send to 11900 penang"))     # 11900  — correct
print(find_postcode("i live at 1190"))           # None   — correctly rejected
print(find_postcode("my order is ORD-48210"))    # ???    — run it and see
```

That third line is your near-miss. Work out what it prints and why, then write
down whether you can fix it with the four pieces of notation you know.
